# 12. Text-to-Speech Synthesis — Save to WAV File (Speech SDK)

This notebook uses the Azure AI Speech SDK's `SpeechSynthesizer` to convert plain text into spoken audio using
a neural voice, saving the result to a WAV file (`cloudxeus_support_message.wav`) instead of playing it through
speakers. This is a fully self-contained, file-in/file-out example — no microphone required, safe to run
directly inside a notebook.

**Difficulty:** Beginner


## Prerequisites

**Python packages:**
- `azure-cognitiveservices-speech` — **not yet in the repo's root `requirements.txt`**, install with:
  ```
  pip3 install azure-cognitiveservices-speech
  ```
- `python-dotenv` (already in `requirements.txt`)

**Azure resources required:**
- An Azure AI Speech resource with a key and endpoint

**Environment variables** (add to root `.env`):
```
AZURE_SPEECH_ENDPOINT=https://<your-speech-resource>.cognitiveservices.azure.com/
AZURE_SPEECH_KEY=<your-speech-resource-key>
```


## What You'll Learn

- How to configure a **neural voice** for text-to-speech synthesis
- How to write synthesized audio to a WAV file instead of playing it live
- How to check the synthesis result's `.reason` to confirm success vs. failure
- Where SSML fits in as a more expressive alternative to plain-text synthesis


### Step 1 — Configure the Speech SDK and select a neural voice

Same `SpeechConfig(subscription=..., endpoint=...)` setup as notebook 11, but this time we also set
`speech_synthesis_voice_name` — here, `"en-US-JennyNeural"`, one of Azure AI Speech's prebuilt **neural**
voices.

💡 **Exam tip:** Azure AI Speech's **neural voices** (named like `en-US-JennyNeural`, `en-US-GuyNeural`, etc.)
produce significantly more natural-sounding, human-like speech than the older, deprecated standard/non-neural
voices — for AI-102 know that neural is the current, recommended voice technology, and that you select a voice
by name via `speech_synthesis_voice_name` (or per-utterance via SSML, see the alternatives note below).

🔄 **Alternatives:** Use **Custom Neural Voice** if you need a synthetic voice that matches a specific brand
identity/persona rather than one of the prebuilt neural voices — this requires training with recorded voice
samples in Speech Studio, a separate, more involved offering from picking a prebuilt voice name.


In [ ]:
import os
from dotenv import load_dotenv
import azure.cognitiveservices.speech as speechsdk

load_dotenv()

endpoint = os.environ["AZURE_SPEECH_ENDPOINT"]
api_key = os.environ["AZURE_SPEECH_KEY"]

speech_config = speechsdk.SpeechConfig(
    subscription=api_key,
    endpoint=endpoint
)

speech_config.speech_synthesis_voice_name = "en-US-JennyNeural"

### Step 2 — Configure file output and synthesize the text

`AudioOutputConfig(filename=...)` routes the synthesized audio to a WAV file on disk instead of the default
speaker output. `SpeechSynthesizer(speech_config, audio_config)` ties the voice configuration and output
destination together. `speak_text_async(text).get()` kicks off synthesis and blocks until it completes,
returning a result object.

💡 **Exam tip:** `speak_text_async()` synthesizes **plain text** — for control over pronunciation, pauses,
emphasis, speaking rate/pitch, or mixing multiple voices in one utterance, use **SSML** (Speech Synthesis
Markup Language) via `speak_ssml_async()` instead. AI-102 expects you to know SSML is the mechanism for that
finer-grained control, not a `speak_text_async()` parameter.

🔄 **Alternatives:** Use `AudioOutputConfig(use_default_speaker=True)` instead of `filename=...` to play the
synthesized speech through your local speakers immediately rather than saving to a file.


In [ ]:
audio_output = speechsdk.audio.AudioOutputConfig(
    filename="cloudxeus_support_message.wav"
)

text = """
Hello, and thank you for contacting CloudXeus Technology Services.
Your support request has been received.
One of our cloud support specialists will review the issue and contact you shortly.
"""

speech_synthesizer = speechsdk.SpeechSynthesizer(
    speech_config=speech_config,
    audio_config=audio_output
)

result = speech_synthesizer.speak_text_async(text).get()

### Step 3 — Check the synthesis result

`result.reason` is a `speechsdk.ResultReason` enum value — comparing it against
`ResultReason.SynthesizingAudioCompleted` confirms synthesis succeeded before reporting success. Production
code should also handle `ResultReason.Canceled` and inspect `result.cancellation_details` for the failure
reason (e.g. auth error, quota, network issue) — this script only checks the success path.

💡 **Exam tip:** Checking `.reason` against a `ResultReason` enum after an async Speech SDK call (`.get()`) is
the standard success/failure pattern across both recognition (`ResultReason.RecognizedSpeech`,
`ResultReason.NoMatch`, `ResultReason.Canceled`) and synthesis (`ResultReason.SynthesizingAudioCompleted`,
`ResultReason.Canceled`) operations — recognize the pattern, not just this specific enum value.

🔄 **Alternatives:** Read `result.audio_data` (raw audio bytes) directly, or use an `AudioDataStream` wrapper,
if you need to work with the synthesized audio in memory instead of only writing it to a file.


In [ ]:
if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    print("Speech synthesis completed successfully.")
    print("Audio file saved as cloudxeus_support_message.wav")

## Summary

This notebook synthesized a short support message into natural-sounding speech using the `en-US-JennyNeural`
neural voice, writing the output to `cloudxeus_support_message.wav` rather than playing it live. This
file-in/file-out pattern is safe to run in any environment, including without local audio hardware, since
synthesis output goes straight to disk.


## Try It Yourself

1. **Easy:** Change `speech_synthesis_voice_name` to a different neural voice (e.g. `"en-US-GuyNeural"` or a
   non-English voice like `"fr-FR-DeniseNeural"`) and compare the resulting audio.
2. **Intermediate:** Rewrite the synthesis call using `speak_ssml_async()` with an SSML string that adds a
   `<break time="500ms"/>` pause and a `<prosody rate="slow">` section around part of the message.
3. **Advanced:** Feed the WAV file this notebook produces into notebook 10's batch transcription client, and
   check whether the transcribed text round-trips back to (approximately) the original message.
